# `assign_eia` — bridge EIA staging → `oil_network` and bind authoritative variables

Per-source assignment notebook. Reads the curated mapping in [production_map.md](production_map.md) and:

1. **Registers** each EIA series we care about in `oil_network.timeseries` (catalogue), linked to an `asset_id` (and optionally a `related_asset_id` for asset-pair series like cross-border outflows). Stores scaling metadata in `attributes.source_scale_to_kbd` so downstream consumers know units are normalised to **kbd** (thousand barrels per day).
2. **Copies** vintaged facts from `oil_network_data_loader.timeseries` into `oil_network.timeseries_data`, applying the source-unit → kbd scale where required (STEO is in million b/d = ×1000 to kbd; `crpdn` MBBL/D is already kbd).
3. **Writes** authoritative `variable_assignments` rows for the production-side TS-bound variables under scenario `starter_us_crude_2015_2025`. Series whose role is *auxiliary* (cross-checks, formula inputs) are registered in steps 1+2 but get **no** `variable_assignments` row — per principle 2.8, observed_auxiliary data is stored without participating in the mass balance.

Idempotent: ON-CONFLICT-UPDATE everywhere. Safe to re-run.

In [1]:
import os
from datetime import datetime

import pandas as pd
from sqlalchemy import create_engine, text

PG_HOST = os.environ.get("PG_HOST", "localhost")
PG_PORT = os.environ.get("PG_PORT", "5432")
PG_DB   = os.environ.get("PG_DB",   "eia_crude")
PG_USER = os.environ.get("PG_USER", "eia_user")
PG_PASS = os.environ.get("PG_PASS", "eia_password")
PG_URL  = f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"

# search_path must include `oil_network` because the variable_assignments
# CHECK trigger function (`variable_assignments_check_graph`) references the
# `scenarios` table without schema-qualification. Until that trigger is
# patched in build_oil_network.ipynb, downstream notebooks must set the path.
engine = create_engine(
    PG_URL,
    connect_args={"options": "-csearch_path=oil_network,public"},
    future=True,
)

SCENARIO_ID    = "starter_us_crude_2015_2025"
EFFECTIVE_FROM = "2015-01-01"
SOURCE         = "eia"


def debug(msg):
    print(f"[{datetime.now():%H:%M:%S}] {msg}")


with engine.connect() as conn:
    db, usr = conn.execute(text("SELECT current_database(), current_user")).one()
debug(f"Connected: {db} as {usr}")
debug(f"Scenario:  {SCENARIO_ID}")
debug(f"Effective: {EFFECTIVE_FROM}")

[17:20:26] Connected: eia_crude as eia_user
[17:20:26] Scenario:  starter_us_crude_2015_2025
[17:20:26] Effective: 2015-01-01


## 1. Mappings — production-side EIA → asset graph

Each row of `EIA_SERIES` describes one EIA series we want in `oil_network.timeseries`:

- `source_id` — the EIA series id as it lives in `oil_network_data_loader.timeseries.timeseries_id`
- `ts_id` — the canonical id we assign in `oil_network.timeseries` (prefixed `eia:` to keep the namespace clean across future sources)
- `asset_id` / `related_asset_id` — the asset linkage. `related_asset_id` is set for asset-pair series (cross-border outflows)
- `ts_type` — one of the 6 `timeseries_types`: production, consumption, inventory, balancing_item, inflow, outflow
- `name` — human-readable label
- `scale_to_kbd` — multiplier applied at copy time. STEO series (`COPR*`, `PAPR*`) are in million b/d → ×1000. `crpdn` MBBL/D series are already kbd → ×1.
- `role` — `authoritative` (gets a `variable_assignments` row binding `variable_id` to this series) or `auxiliary` (registered for cross-check / formula input only)
- `variable_id` — only for `authoritative`; the variable this series binds to

In [2]:
# EIA-series mapping. Each tuple = (source_id, ts_id, asset_id, related_asset_id,
# ts_type, name, scale_to_kbd, unit, role, variable_id_or_None).
#
# 2026-05-08 zero-gap pass: every loaded EIA series binds to a graph variable.
# 2026-05-08 unit pass: each tuple carries its source unit. For 'mbbl' (monthly
# volume) series we DUAL-REGISTER:
#   - the source as `<ts_id>` with unit='mbbl' (audit/raw)
#   - a derived `<ts_id>_kbd` with unit='kbd', values divided by per-row days_in_month
# variable_assignments always reference the kbd-suffixed (rate-form) version.

EIA_SERIES = [
    # =========================================================================
    # PRODUCTION-SIDE — all in MBBL/D (already kbd, no conversion)
    # =========================================================================
    ("MCRFPNM2", "eia:MCRFPNM2", "permian_nm", None, "production",
        "Permian-NM (NM state) crude production", 1.0, "kbd", "authoritative",
        "production__crude__permian_nm"),
    ("MCRFPND2", "eia:MCRFPND2", "bakken_nd", None, "production",
        "Bakken-ND (ND state) crude production", 1.0, "kbd", "authoritative",
        "production__crude__bakken_nd"),
    ("COPREF", "eia:COPREF", "eagle_ford_tx", None, "production",
        "Eagle Ford crude production (STEO basin = TX state)", 1000.0, "kbd", "authoritative",
        "production__crude__eagle_ford_tx"),
    ("MCRFP3FM2", "eia:MCRFP3FM2", "gulf_of_america", None, "production",
        "Federal-offshore Gulf of America crude production", 1.0, "kbd", "authoritative",
        "production__crude__gulf_of_america"),
    ("MANFPAK2", "eia:MANFPAK2", "alaska_north_slope", None, "production",
        "Alaska North Slope crude production", 1.0, "kbd", "authoritative",
        "production__crude__alaska_north_slope"),
    ("MCRFPCA2", "eia:MCRFPCA2", "california_conventional", None, "production",
        "California state crude production", 1.0, "kbd", "authoritative",
        "production__crude__california_conventional"),
    ("MCRFPOK2", "eia:MCRFPOK2", "oklahoma_conventional", None, "production",
        "Oklahoma state crude production", 1.0, "kbd", "authoritative",
        "production__crude__oklahoma_conventional"),
    ("MCRFPWY2", "eia:MCRFPWY2", "wyoming_conventional", None, "production",
        "Wyoming state crude production", 1.0, "kbd", "authoritative",
        "production__crude__wyoming_conventional"),
    ("MCRFPCO2", "eia:MCRFPCO2", "colorado_conventional", None, "production",
        "Colorado state crude production", 1.0, "kbd", "authoritative",
        "production__crude__colorado_conventional"),

    # Canada cross-border outflows (per-pipeline observed)
    ("MCRIPP5CA2", "eia:MCRIPP5CA2", "canadian_oil_sands", "pipe_trans_mountain_tmx", "outflow",
        "Canada -> PADD 5 imports (TMX 1:1 proxy)", 1.0, "kbd", "authoritative",
        "outflow__crude__canadian_oil_sands__pipe_trans_mountain_tmx"),
    ("MCRIPP4CA2", "eia:MCRIPP4CA2", "canadian_oil_sands", "pipe_express_platte", "outflow",
        "Canada -> PADD 4 imports (Express+Platte 1:1 proxy)", 1.0, "kbd", "authoritative",
        "outflow__crude__canadian_oil_sands__pipe_express_platte"),

    # Basin aggregates TS-bound
    ("COPRPM", "eia:COPRPM", "permian", None, "production",
        "Permian basin crude production (STEO)", 1000.0, "kbd", "authoritative",
        "production__crude__permian"),
    ("COPRBK", "eia:COPRBK", "bakken", None, "production",
        "Bakken basin crude production (STEO)", 1000.0, "kbd", "authoritative",
        "production__crude__bakken"),

    # State aggregates TS-bound
    ("MCRFPTX2", "eia:MCRFPTX2", "texas_state_view", None, "production",
        "Texas state crude production aggregate view", 1.0, "kbd", "authoritative",
        "production__crude__texas_state_view"),
    ("MCRFPMT2", "eia:MCRFPMT2", "montana_state_view", None, "production",
        "Montana state crude production aggregate view", 1.0, "kbd", "authoritative",
        "production__crude__montana_state_view"),

    # PADD production aggregates TS-bound
    ("MCRFPP12", "eia:MCRFPP12", "padd1_view", None, "production",
        "PADD 1 crude production", 1.0, "kbd", "authoritative",
        "production__crude__padd1_view"),
    ("MCRFPP22", "eia:MCRFPP22", "padd2_view", None, "production",
        "PADD 2 crude production", 1.0, "kbd", "authoritative",
        "production__crude__padd2_view"),
    ("MCRFPP32", "eia:MCRFPP32", "padd3_view", None, "production",
        "PADD 3 crude production", 1.0, "kbd", "authoritative",
        "production__crude__padd3_view"),
    ("MCRFPP42", "eia:MCRFPP42", "padd4_view", None, "production",
        "PADD 4 crude production", 1.0, "kbd", "authoritative",
        "production__crude__padd4_view"),
    ("MCRFPP52", "eia:MCRFPP52", "padd5_view", None, "production",
        "PADD 5 crude production", 1.0, "kbd", "authoritative",
        "production__crude__padd5_view"),

    # National aggregates TS-bound
    ("MCRFPUS2", "eia:MCRFPUS2", "usa_view", None, "production",
        "USA total crude production", 1.0, "kbd", "authoritative",
        "production__crude__usa_view"),
    ("PAPR48NGOM", "eia:PAPR48NGOM", "usa_lower48_excl_gom_view", None, "production",
        "USA Lower 48 excl GoM crude production (STEO)", 1000.0, "kbd", "authoritative",
        "production__crude__usa_lower48_excl_gom_view"),

    # =========================================================================
    # IMPORTS — all in MBBL/D (already kbd)
    # =========================================================================
    ("MCRIMUSCA2", "eia:MCRIMUSCA2", "usa_view", 'canadian_oil_sands', "inflow",
        "USA total imports from Canada", 1.0, "kbd", "authoritative",
        "inflow__crude__usa_view__canadian_oil_sands"),
    ("MCRIPP1CA2", "eia:MCRIPP1CA2", "padd1_view", 'canadian_oil_sands', "inflow",
        "PADD 1 imports from Canada", 1.0, "kbd", "authoritative",
        "inflow__crude__padd1_view__canadian_oil_sands"),
    ("MCRIPP2CA2", "eia:MCRIPP2CA2", "padd2_view", 'canadian_oil_sands', "inflow",
        "PADD 2 imports from Canada", 1.0, "kbd", "authoritative",
        "inflow__crude__padd2_view__canadian_oil_sands"),
    ("MCRIPP3CA2", "eia:MCRIPP3CA2", "padd3_view", 'canadian_oil_sands', "inflow",
        "PADD 3 imports from Canada", 1.0, "kbd", "authoritative",
        "inflow__crude__padd3_view__canadian_oil_sands"),
    ("MCRIPP4CA2", "eia:MCRIPP4CA2", "padd4_view", 'canadian_oil_sands', "inflow",
        "PADD 4 imports from Canada (Express+Platte corridor)", 1.0, "kbd", "authoritative",
        "inflow__crude__padd4_view__canadian_oil_sands"),
    ("MCRIPP5CA2", "eia:MCRIPP5CA2", "padd5_view", 'canadian_oil_sands', "inflow",
        "PADD 5 imports from Canada (TMX corridor)", 1.0, "kbd", "authoritative",
        "inflow__crude__padd5_view__canadian_oil_sands"),

    ("MCRIMUS2", "eia:MCRIMUS2", "usa_view", None, "inflow",
        "USA total crude imports (all countries)", 1.0, "kbd", "auxiliary",
        None),
    ("MCRIPP12", "eia:MCRIPP12", "padd1_view", None, "inflow",
        "PADD 1 total crude imports (all countries)", 1.0, "kbd", "auxiliary",
        None),
    ("MCRIPP22", "eia:MCRIPP22", "padd2_view", None, "inflow",
        "PADD 2 total crude imports (all countries)", 1.0, "kbd", "auxiliary",
        None),
    ("MCRIPP32", "eia:MCRIPP32", "padd3_view", None, "inflow",
        "PADD 3 total crude imports (all countries)", 1.0, "kbd", "auxiliary",
        None),
    ("MCRIPP42", "eia:MCRIPP42", "padd4_view", None, "inflow",
        "PADD 4 total crude imports (all countries)", 1.0, "kbd", "auxiliary",
        None),
    ("MCRIPP52", "eia:MCRIPP52", "padd5_view", None, "inflow",
        "PADD 5 total crude imports (all countries)", 1.0, "kbd", "auxiliary",
        None),

    # =========================================================================
    # EXPORTS — all in MBBL/D (already kbd) — added 2026-05-08
    # =========================================================================
    # Per-PADD totals: 4 modelled export terminals are all in PADD 3.
    # PADDs 1/2/4/5 export views are observation-only (no terminals modelled).
    ("MCREXUS2", "eia:MCREXUS2", "usa_view", 'foreign_export_destination', "outflow",
        "USA total crude exports (all destinations)", 1.0, "kbd", "authoritative",
        "outflow__crude__usa_view__foreign_export_destination"),
    ("MCREXP12", "eia:MCREXP12", "padd1_view", 'foreign_export_destination', "outflow",
        "PADD 1 total crude exports", 1.0, "kbd", "authoritative",
        "outflow__crude__padd1_view__foreign_export_destination"),
    ("MCREXP22", "eia:MCREXP22", "padd2_view", 'foreign_export_destination', "outflow",
        "PADD 2 total crude exports", 1.0, "kbd", "authoritative",
        "outflow__crude__padd2_view__foreign_export_destination"),
    ("MCREXP32", "eia:MCREXP32", "padd3_view", 'foreign_export_destination', "outflow",
        "PADD 3 total crude exports (modelled terminals)", 1.0, "kbd", "authoritative",
        "outflow__crude__padd3_view__foreign_export_destination"),
    ("MCREXP42", "eia:MCREXP42", "padd4_view", 'foreign_export_destination', "outflow",
        "PADD 4 total crude exports", 1.0, "kbd", "authoritative",
        "outflow__crude__padd4_view__foreign_export_destination"),
    ("MCREXP52", "eia:MCREXP52", "padd5_view", 'foreign_export_destination', "outflow",
        "PADD 5 total crude exports", 1.0, "kbd", "authoritative",
        "outflow__crude__padd5_view__foreign_export_destination"),

    # =========================================================================
    # INTER-PADD MOVEMENTS — MBBL (volume, monthly) — DUAL-REGISTERED
    # =========================================================================
    ("MCRMPP1P21", "eia:MCRMPP1P21", "padd2_view", "padd1_view", "outflow",
        "Inter-PADD movement: PADD 2 -> PADD 1", 1.0, "mbbl", "auxiliary",
        None),  # raw pipeline-only; superseded by combined inter-PADD series below
    ("MCRMPP2P11", "eia:MCRMPP2P11", "padd1_view", "padd2_view", "outflow",
        "Inter-PADD movement: PADD 1 -> PADD 2", 1.0, "mbbl", "auxiliary",
        None),  # raw pipeline-only; superseded by combined inter-PADD series below
    ("MCRMPP2P31", "eia:MCRMPP2P31", "padd3_view", "padd2_view", "outflow",
        "Inter-PADD movement: PADD 3 -> PADD 2", 1.0, "mbbl", "auxiliary",
        None),  # raw pipeline-only; superseded by combined inter-PADD series below
    ("MCRMPP2P41", "eia:MCRMPP2P41", "padd4_view", "padd2_view", "outflow",
        "Inter-PADD movement: PADD 4 -> PADD 2", 1.0, "mbbl", "auxiliary",
        None),  # raw pipeline-only; superseded by combined inter-PADD series below
    ("MCRMPP3P11", "eia:MCRMPP3P11", "padd1_view", "padd3_view", "outflow",
        "Inter-PADD movement: PADD 1 -> PADD 3", 1.0, "mbbl", "auxiliary",
        None),  # raw pipeline-only; superseded by combined inter-PADD series below
    ("MCRMPP3P21", "eia:MCRMPP3P21", "padd2_view", "padd3_view", "outflow",
        "Inter-PADD movement: PADD 2 -> PADD 3", 1.0, "mbbl", "auxiliary",
        None),  # raw pipeline-only; superseded by combined inter-PADD series below
    ("MCRMPP3P41", "eia:MCRMPP3P41", "padd4_view", "padd3_view", "outflow",
        "Inter-PADD movement: PADD 4 -> PADD 3", 1.0, "mbbl", "auxiliary",
        None),  # raw pipeline-only; superseded by combined inter-PADD series below
    ("MCRMPP4P21", "eia:MCRMPP4P21", "padd2_view", "padd4_view", "outflow",
        "Inter-PADD movement: PADD 2 -> PADD 4", 1.0, "mbbl", "auxiliary",
        None),  # raw pipeline-only; superseded by combined inter-PADD series below

    # =========================================================================
    # REFINING-SIDE — all in MBBL/D
    # =========================================================================
    ("M_EPC0_YIY_REC_2", "eia:M_EPC0_YIY_REC_2", "district_REC_refining_view", None, "consumption",
        "PADD 1 East Coast refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_REC_refining_view"),
    ("M_EPC0_YIY_RAP_2", "eia:M_EPC0_YIY_RAP_2", "district_RAP_refining_view", None, "consumption",
        "PADD 1 Appalachian No.1 refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_RAP_refining_view"),
    ("M_EPC0_YIY_R2A_2", "eia:M_EPC0_YIY_R2A_2", "district_R2A_refining_view", None, "consumption",
        "PADD 2 Indiana-Illinois-Kentucky refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_R2A_refining_view"),
    ("M_EPC0_YIY_R2B_2", "eia:M_EPC0_YIY_R2B_2", "district_R2B_refining_view", None, "consumption",
        "PADD 2 Minnesota-Wisconsin-North/South Dakota refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_R2B_refining_view"),
    ("M_EPC0_YIY_R2C_2", "eia:M_EPC0_YIY_R2C_2", "district_R2C_refining_view", None, "consumption",
        "PADD 2 Oklahoma-Kansas-Missouri refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_R2C_refining_view"),
    ("M_EPC0_YIY_R3A_2", "eia:M_EPC0_YIY_R3A_2", "district_R3A_refining_view", None, "consumption",
        "PADD 3 Texas Inland refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_R3A_refining_view"),
    ("M_EPC0_YIY_R3B_2", "eia:M_EPC0_YIY_R3B_2", "district_R3B_refining_view", None, "consumption",
        "PADD 3 Texas Gulf Coast refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_R3B_refining_view"),
    ("M_EPC0_YIY_R3C_2", "eia:M_EPC0_YIY_R3C_2", "district_R3C_refining_view", None, "consumption",
        "PADD 3 Louisiana Gulf Coast refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_R3C_refining_view"),
    ("M_EPC0_YIY_R3D_2", "eia:M_EPC0_YIY_R3D_2", "district_R3D_refining_view", None, "consumption",
        "PADD 3 N. Louisiana-Arkansas refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_R3D_refining_view"),
    ("M_EPC0_YIY_R3E_2", "eia:M_EPC0_YIY_R3E_2", "district_R3E_refining_view", None, "consumption",
        "PADD 3 New Mexico refining district crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__district_R3E_refining_view"),
    ("M_EPC0_YIY_R40_2", "eia:M_EPC0_YIY_R40_2", "padd4_view", None, "consumption",
        "PADD 4 (Rocky Mountain) refining crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__padd4_view"),
    ("M_EPC0_YIY_R50_2", "eia:M_EPC0_YIY_R50_2", "padd5_view", None, "consumption",
        "PADD 5 (West Coast) refining crude inputs", 1.0, "kbd", "authoritative",
        "consumption__crude__padd5_view"),

    ("M_EPC0_YIY_R10_2", "eia:M_EPC0_YIY_R10_2", "padd1_view", None, "consumption",
        "PADD 1 refinery net input of crude", 1.0, "kbd", "authoritative",
        "consumption__crude__padd1_view"),
    ("M_EPC0_YIY_R20_2", "eia:M_EPC0_YIY_R20_2", "padd2_view", None, "consumption",
        "PADD 2 refinery net input of crude", 1.0, "kbd", "authoritative",
        "consumption__crude__padd2_view"),
    ("M_EPC0_YIY_R30_2", "eia:M_EPC0_YIY_R30_2", "padd3_view", None, "consumption",
        "PADD 3 refinery net input of crude", 1.0, "kbd", "authoritative",
        "consumption__crude__padd3_view"),

    ("M_EPC0_YIY_NUS_2", "eia:M_EPC0_YIY_NUS_2", "usa_view", None, "consumption",
        "USA total refinery net input of crude", 1.0, "kbd", "authoritative",
        "consumption__crude__usa_view"),

    # =========================================================================
    # STOCKS / INVENTORY -- MBBL ending-month levels (snapshots, NOT rates)
    # unit='mbbl_level' tags these as levels; no per-row days conversion.
    # =========================================================================
    # Cushing hub: only EIA-tracked individual hub.
    ("MCRST_YCUOK_1", "eia:MCRST_YCUOK_1", "cushing_hub", None, "inventory",
        "Cushing OK crude ending stocks", 1.0, "mbbl_level", "authoritative",
        "inventory__crude__cushing_hub"),

    # National
    ("MCRSTUS1", "eia:MCRSTUS1", "usa_view", None, "inventory",
        "USA total crude ending stocks (commercial + SPR + in-transit)", 1.0, "mbbl_level", "authoritative",
        "inventory__crude__usa_view"),
    ("MCESTUS1", "eia:MCESTUS1", "usa_view", None, "inventory",
        "USA commercial crude ending stocks (excl. SPR)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCSSTUS1", "eia:MCSSTUS1", "spr_total", None, "inventory",
        "USA SPR crude ending stocks", 1.0, "mbbl_level", "authoritative",
        "inventory__crude__spr_total"),

    # PADD-level
    ("MCRSTP11", "eia:MCRSTP11", "padd1_view", None, "inventory",
        "PADD 1 crude ending stocks", 1.0, "mbbl_level", "authoritative",
        "inventory__crude__padd1_view"),
    ("MCRSTP21", "eia:MCRSTP21", "padd2_view", None, "inventory",
        "PADD 2 crude ending stocks (incl. Cushing)", 1.0, "mbbl_level", "authoritative",
        "inventory__crude__padd2_view"),
    ("MCRSTP31", "eia:MCRSTP31", "padd3_view", None, "inventory",
        "PADD 3 crude ending stocks", 1.0, "mbbl_level", "authoritative",
        "inventory__crude__padd3_view"),
    ("MCRSTP41", "eia:MCRSTP41", "padd4_view", None, "inventory",
        "PADD 4 crude ending stocks", 1.0, "mbbl_level", "authoritative",
        "inventory__crude__padd4_view"),
    ("MCRSTP51", "eia:MCRSTP51", "padd5_view", None, "inventory",
        "PADD 5 crude ending stocks", 1.0, "mbbl_level", "authoritative",
        "inventory__crude__padd5_view"),

    # PADD-level stock decomposition: Tank Farms + Pipelines vs Refinery operating stocks.
    # AUXILIARY: EIA publishes these at PADD-level only (no finer breakdown), so we cannot
    # add structural sub-nodes without double-counting Cushing/Patoka (already members of
    # the tank-farms+pipelines aggregate). Stored as observed_auxiliary on padd_view per
    # Principle 2.8 — cross-check data, not part of the mass balance at this level.
    # Verified identity: MCRSFP{N}1 + MCRRSP{N}1 = MCRSTP{N}1.
    ("MCRSFP11", "eia:MCRSFP11", "padd1_view", None, "inventory",
        "PADD 1 crude stocks at Tank Farms + Pipelines (decomposition of MCRSTP11)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCRSFP21", "eia:MCRSFP21", "padd2_view", None, "inventory",
        "PADD 2 crude stocks at Tank Farms + Pipelines (decomposition of MCRSTP21)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCRSFP31", "eia:MCRSFP31", "padd3_view", None, "inventory",
        "PADD 3 crude stocks at Tank Farms + Pipelines (decomposition of MCRSTP31)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCRSFP41", "eia:MCRSFP41", "padd4_view", None, "inventory",
        "PADD 4 crude stocks at Tank Farms + Pipelines (decomposition of MCRSTP41)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCRSFP51", "eia:MCRSFP51", "padd5_view", None, "inventory",
        "PADD 5 crude stocks at Tank Farms + Pipelines (decomposition of MCRSTP51)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCRRSP11", "eia:MCRRSP11", "padd1_view", None, "inventory",
        "PADD 1 crude stocks at Refineries (decomposition of MCRSTP11)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCRRSP21", "eia:MCRRSP21", "padd2_view", None, "inventory",
        "PADD 2 crude stocks at Refineries (decomposition of MCRSTP21)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCRRSP31", "eia:MCRRSP31", "padd3_view", None, "inventory",
        "PADD 3 crude stocks at Refineries (decomposition of MCRSTP31)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCRRSP41", "eia:MCRRSP41", "padd4_view", None, "inventory",
        "PADD 4 crude stocks at Refineries (decomposition of MCRSTP41)", 1.0, "mbbl_level", "auxiliary",
        None),
    ("MCRRSP51", "eia:MCRRSP51", "padd5_view", None, "inventory",
        "PADD 5 crude stocks at Refineries (decomposition of MCRSTP51)", 1.0, "mbbl_level", "auxiliary",
        None),

    # =========================================================================
    # SUPPLY ADJUSTMENT (balancing item) — EIA's own observed B, in MBBL/D (kbd)
    # Principle 2.8: TS-observed_authoritative — promotes B from derived-closure
    # to observed. The closure formula B = ΔS − P + C − ΣI + ΣO becomes a
    # constraint cross-check (Chapter 5 Claim 4) rather than the primary resolver.
    # Must remove the corresponding closure-formula tuples from assign_formulas.
    # =========================================================================
    ("MCRUA_NUS_2", "eia:MCRUA_NUS_2", "usa_view", None, "balancing_item",
        "USA Supply Adjustment of Crude Oil (EIA observed B)", 1.0, "kbd", "authoritative",
        "balancing_item__crude__usa_view"),
    ("MCRUA_R10_2", "eia:MCRUA_R10_2", "padd1_view", None, "balancing_item",
        "PADD 1 Supply Adjustment of Crude Oil (EIA observed B)", 1.0, "kbd", "authoritative",
        "balancing_item__crude__padd1_view"),
    ("MCRUA_R20_2", "eia:MCRUA_R20_2", "padd2_view", None, "balancing_item",
        "PADD 2 Supply Adjustment of Crude Oil (EIA observed B)", 1.0, "kbd", "authoritative",
        "balancing_item__crude__padd2_view"),
    ("MCRUA_R30_2", "eia:MCRUA_R30_2", "padd3_view", None, "balancing_item",
        "PADD 3 Supply Adjustment of Crude Oil (EIA observed B)", 1.0, "kbd", "authoritative",
        "balancing_item__crude__padd3_view"),
    ("MCRUA_R40_2", "eia:MCRUA_R40_2", "padd4_view", None, "balancing_item",
        "PADD 4 Supply Adjustment of Crude Oil (EIA observed B)", 1.0, "kbd", "authoritative",
        "balancing_item__crude__padd4_view"),
    ("MCRUA_R50_2", "eia:MCRUA_R50_2", "padd5_view", None, "balancing_item",
        "PADD 5 Supply Adjustment of Crude Oil (EIA observed B)", 1.0, "kbd", "authoritative",
        "balancing_item__crude__padd5_view"),

    # =========================================================================
    # INTER-PADD MOVEMENTS — raw component series (tanker/barge + sub-PADD pipeline)
    # All in MBBL (monthly volume). Variable binding happens via combined kbd series
    # built in the cell below.
    # =========================================================================
    # Tanker/barge components per direction (10 series)
    ("MCRMTP1P21", "eia:MCRMTP1P21", "padd2_view", "padd1_view", "outflow",
        "Inter-PADD movement P1->P2 by tanker/barge", 1.0, "mbbl", "auxiliary", None),
    ("MCRMTP1P31", "eia:MCRMTP1P31", "padd3_view", "padd1_view", "outflow",
        "Inter-PADD movement P1->P3 by tanker/barge", 1.0, "mbbl", "auxiliary", None),
    ("MCRMTP2P11", "eia:MCRMTP2P11", "padd1_view", "padd2_view", "outflow",
        "Inter-PADD movement P2->P1 by tanker/barge", 1.0, "mbbl", "auxiliary", None),
    ("MCRMTP2P31", "eia:MCRMTP2P31", "padd3_view", "padd2_view", "outflow",
        "Inter-PADD movement P2->P3 by tanker/barge", 1.0, "mbbl", "auxiliary", None),
    ("MCRMTP3P11", "eia:MCRMTP3P11", "padd1_view", "padd3_view", "outflow",
        "Inter-PADD movement P3->P1 by tanker/barge", 1.0, "mbbl", "auxiliary", None),
    ("MCRMTP3P21", "eia:MCRMTP3P21", "padd2_view", "padd3_view", "outflow",
        "Inter-PADD movement P3->P2 by tanker/barge", 1.0, "mbbl", "auxiliary", None),
    ("MCRMTP3P51", "eia:MCRMTP3P51", "padd5_view", "padd3_view", "outflow",
        "Inter-PADD movement P3->P5 by tanker/barge (Jones Act WB->WC)", 1.0, "mbbl", "auxiliary", None),
    ("MCRMTP5P31", "eia:MCRMTP5P31", "padd3_view", "padd5_view", "outflow",
        "Inter-PADD movement P3->P5 by tanker/barge", 1.0, "mbbl", "auxiliary", None),
    ("MCRMT1YP31", "eia:MCRMT1YP31", "padd1_view", "padd3_view", "outflow",
        "Inter-PADD movement R1Y->P3 by tanker/barge (sub-PADD 1 Atlantic origin)", 1.0, "mbbl", "auxiliary", None),
    ("MCRMT1ZP31", "eia:MCRMT1ZP31", "padd1_view", "padd3_view", "outflow",
        "Inter-PADD movement R1Z->P3 by tanker/barge (sub-PADD 1 Atlantic origin, sparse)", 1.0, "mbbl", "auxiliary", None),

    # Sub-PADD pipeline components for P1->P3 and P4->P3 (no top-level MCRMP_P1_P3 etc)
    ("MCRMP_R10-R30_1", "eia:MCRMP_R10-R30_1", "padd3_view", "padd1_view", "outflow",
        "Inter-PADD movement P1->P3 by pipeline (R10-R30 sub-PADD)", 1.0, "mbbl", "auxiliary", None),
    ("MCRMP_R40-R30_1", "eia:MCRMP_R40-R30_1", "padd3_view", "padd4_view", "outflow",
        "Inter-PADD movement P4->P3 by pipeline (R40-R30 sub-PADD)", 1.0, "mbbl", "auxiliary", None),
    # =========================================================================
    # REGION-AGGREGATE FOREIGN-SUPPLY INFLOWS (derived series: total - Canadian)
    # =========================================================================
    # USA: foreign_supply inflow = MCRIMUS2 - MCRIMUSCA2 (non-Canadian)
    ("FOREIGN_SUPPLY_USA", "eia:foreign_supply_to_usa_kbd", "usa_view", "foreign_supply", "inflow",
        "USA non-Canadian foreign crude imports (derived: MCRIMUS2 - MCRIMUSCA2)",
        1.0, "kbd", "authoritative",
        "inflow__crude__usa_view__foreign_supply"),
    # PADD 1: MCRIPP12 - MCRIPP1CA2
    ("FOREIGN_SUPPLY_P1", "eia:foreign_supply_to_padd1_kbd", "padd1_view", "foreign_supply", "inflow",
        "PADD 1 non-Canadian foreign crude imports (derived)",
        1.0, "kbd", "authoritative",
        "inflow__crude__padd1_view__foreign_supply"),
    # PADD 2: MCRIPP22 - MCRIPP2CA2
    ("FOREIGN_SUPPLY_P2", "eia:foreign_supply_to_padd2_kbd", "padd2_view", "foreign_supply", "inflow",
        "PADD 2 non-Canadian foreign crude imports (derived)",
        1.0, "kbd", "authoritative",
        "inflow__crude__padd2_view__foreign_supply"),
    # PADD 3: MCRIPP32 - MCRIPP3CA2
    ("FOREIGN_SUPPLY_P3", "eia:foreign_supply_to_padd3_kbd", "padd3_view", "foreign_supply", "inflow",
        "PADD 3 non-Canadian foreign crude imports (derived)",
        1.0, "kbd", "authoritative",
        "inflow__crude__padd3_view__foreign_supply"),
    # PADD 4: MCRIPP42 - MCRIPP4CA2
    ("FOREIGN_SUPPLY_P4", "eia:foreign_supply_to_padd4_kbd", "padd4_view", "foreign_supply", "inflow",
        "PADD 4 non-Canadian foreign crude imports (derived)",
        1.0, "kbd", "authoritative",
        "inflow__crude__padd4_view__foreign_supply"),
    # PADD 5: MCRIPP52 - MCRIPP5CA2
    ("FOREIGN_SUPPLY_P5", "eia:foreign_supply_to_padd5_kbd", "padd5_view", "foreign_supply", "inflow",
        "PADD 5 non-Canadian foreign crude imports (derived)",
        1.0, "kbd", "authoritative",
        "inflow__crude__padd5_view__foreign_supply"),
]

df_map = pd.DataFrame(EIA_SERIES, columns=[
    "source_id", "ts_id", "asset_id", "related_asset_id", "ts_type",
    "name", "scale_to_kbd", "unit", "role", "variable_id",
])
debug(f"Mapping: {len(df_map)} EIA series  ({(df_map['role']=='authoritative').sum()} authoritative, {(df_map['role']=='auxiliary').sum()} auxiliary)")
debug(f"  by source unit: {df_map['unit'].value_counts().to_dict()}")

[17:20:26] Mapping: 107 EIA series  (70 authoritative, 37 auxiliary)
[17:20:26]   by source unit: {'kbd': 68, 'mbbl': 20, 'mbbl_level': 19}


## 2. Register timeseries (catalogue)

UPSERT into `oil_network.timeseries`. The `attributes` JSONB stores provenance metadata (source series id, scale factor, role) so anyone querying the catalogue can see exactly where the value came from and how it was normalised.

In [3]:
import json as _json

rows_for_ts = []
for r in EIA_SERIES:
    src_id, ts_id, asset_id, rel_asset_id, ts_type, name, scale, unit, role, _var = r
    # 1) Register the source series exactly as published.
    attrs_raw = {
        "source_series_id": src_id,
        "source_dataset": "eia",
        "scale_to_kbd": scale,
        "role": role,
        "kind": "raw",
    }
    rows_for_ts.append({
        "timeseries_id": ts_id,
        "name": name + (f" [raw {unit}]" if unit != "kbd" else ""),
        "description": name,
        "timeseries_type": ts_type,
        "commodity": "crude",
        "asset_id": asset_id,
        "related_asset_id": rel_asset_id,
        "source": SOURCE,
        "unit": unit,
        "attributes": _json.dumps(attrs_raw),
    })

    # 2) For mbbl (monthly volume) sources, additionally register a derived kbd
    #    rate-form series. The data copy step handles the per-row conversion.
    #    For mbbl_level (snapshot) sources, no derived rate is created -- they
    #    are inventory levels, not flows.
    if unit == "mbbl":
        ts_id_kbd = f"{ts_id}_kbd"
        attrs_kbd = {
            "source_series_id": src_id,
            "source_dataset": "eia",
            "scale_to_kbd": scale,
            "role": role,
            "kind": "derived_kbd_rate",
            "derived_from": ts_id,
            "derivation": "value_mbbl / days_in_month(observation_date)",
        }
        rows_for_ts.append({
            "timeseries_id": ts_id_kbd,
            "name": name + " [kbd, per-row days conversion]",
            "description": name + " — derived rate from raw monthly volume",
            "timeseries_type": ts_type,
            "commodity": "crude",
            "asset_id": asset_id,
            "related_asset_id": rel_asset_id,
            "source": SOURCE,
            "unit": "kbd",
            "attributes": _json.dumps(attrs_kbd),
        })

UPSERT_TS = text("""
    INSERT INTO oil_network.timeseries
        (timeseries_id, name, description, timeseries_type, commodity,
         asset_id, related_asset_id, source, unit, attributes)
    VALUES
        (:timeseries_id, :name, :description, :timeseries_type, :commodity,
         :asset_id, :related_asset_id, :source, :unit, CAST(:attributes AS JSONB))
    ON CONFLICT (timeseries_id) DO UPDATE SET
        name             = EXCLUDED.name,
        description      = EXCLUDED.description,
        timeseries_type  = EXCLUDED.timeseries_type,
        commodity        = EXCLUDED.commodity,
        asset_id         = EXCLUDED.asset_id,
        related_asset_id = EXCLUDED.related_asset_id,
        source           = EXCLUDED.source,
        unit             = EXCLUDED.unit,
        attributes       = EXCLUDED.attributes
""")

with engine.begin() as conn:
    conn.execute(UPSERT_TS, rows_for_ts)

n_total = pd.read_sql(text("SELECT COUNT(*) FROM oil_network.timeseries WHERE source = 'eia'"), engine).iloc[0, 0]
n_kbd = pd.read_sql(text("SELECT COUNT(*) FROM oil_network.timeseries WHERE source='eia' AND unit='kbd'"), engine).iloc[0, 0]
n_mbbl = pd.read_sql(text("SELECT COUNT(*) FROM oil_network.timeseries WHERE source='eia' AND unit='mbbl'"), engine).iloc[0, 0]
n_lev = pd.read_sql(text("SELECT COUNT(*) FROM oil_network.timeseries WHERE source='eia' AND unit='mbbl_level'"), engine).iloc[0, 0]
debug(f"oil_network.timeseries (source='eia'): {n_total} rows  (kbd={n_kbd}, mbbl={n_mbbl}, mbbl_level={n_lev})")

[17:20:26] oil_network.timeseries (source='eia'): 137 rows  (kbd=98, mbbl=20, mbbl_level=19)


## 3. Copy vintaged facts → `oil_network.timeseries_data`

For each registered EIA series, copy `oil_network_data_loader.timeseries` rows into `oil_network.timeseries_data`, applying the scale and renaming the timeseries-id (prepending `eia:`). The vintage timestamp is preserved as `saved_date`. The PK `(timeseries_id, observation_date, saved_date)` makes this idempotent — re-runs are safe and accumulate vintages naturally if the staging schema accumulates them.

In [4]:
# Two SQL templates: one for kbd-already sources (constant scale), one for
# mbbl sources (per-row days-in-month conversion).
COPY_FACTS_KBD = text("""
    INSERT INTO oil_network.timeseries_data
        (timeseries_id, observation_date, saved_date, value)
    SELECT
        :ts_id,
        s.timeseries_date::date,
        s.timeseries_published_date::date,
        s.timeseries_value * :scale
    FROM oil_network_data_loader.timeseries s
    WHERE s.source = 'eia'
      AND s.timeseries_id = :source_id
    ON CONFLICT (timeseries_id, observation_date, saved_date) DO UPDATE
        SET value = EXCLUDED.value
""")

# For mbbl sources we copy raw values to <ts_id> AND, in a second insert, copy
# values divided by the actual number of days in the month to <ts_id>_kbd.
# date_trunc + interval arithmetic gives the exact days-in-month for the row.
COPY_FACTS_MBBL_RAW = text("""
    INSERT INTO oil_network.timeseries_data
        (timeseries_id, observation_date, saved_date, value)
    SELECT
        :ts_id,
        s.timeseries_date::date,
        s.timeseries_published_date::date,
        s.timeseries_value
    FROM oil_network_data_loader.timeseries s
    WHERE s.source = 'eia'
      AND s.timeseries_id = :source_id
    ON CONFLICT (timeseries_id, observation_date, saved_date) DO UPDATE
        SET value = EXCLUDED.value
""")

COPY_FACTS_MBBL_DERIVED_KBD = text("""
    INSERT INTO oil_network.timeseries_data
        (timeseries_id, observation_date, saved_date, value)
    SELECT
        :ts_id_kbd,
        s.timeseries_date::date,
        s.timeseries_published_date::date,
        s.timeseries_value
            / EXTRACT(DAY FROM (date_trunc('month', s.timeseries_date::date) + interval '1 month - 1 day'))
    FROM oil_network_data_loader.timeseries s
    WHERE s.source = 'eia'
      AND s.timeseries_id = :source_id
    ON CONFLICT (timeseries_id, observation_date, saved_date) DO UPDATE
        SET value = EXCLUDED.value
""")

total_inserted = 0
for r in EIA_SERIES:
    src_id, ts_id, _ai, _rai, _tt, _name, scale, unit, _role, _v = r
    if unit == "kbd":
        with engine.begin() as conn:
            res = conn.execute(COPY_FACTS_KBD, {"ts_id": ts_id, "source_id": src_id, "scale": scale})
            total_inserted += res.rowcount
    elif unit == "mbbl":
        ts_id_kbd = f"{ts_id}_kbd"
        with engine.begin() as conn:
            r1 = conn.execute(COPY_FACTS_MBBL_RAW, {"ts_id": ts_id, "source_id": src_id})
            r2 = conn.execute(COPY_FACTS_MBBL_DERIVED_KBD, {"ts_id_kbd": ts_id_kbd, "source_id": src_id})
            total_inserted += r1.rowcount + r2.rowcount
    elif unit == "mbbl_level":
        # Stock LEVEL (snapshot in MBBL) -- copy unchanged, no rate derivation.
        # Same SQL shape as MBBL_RAW (value passes through), but semantics differ:
        # the value IS the inventory at end-of-month, not a monthly volume.
        with engine.begin() as conn:
            res = conn.execute(COPY_FACTS_MBBL_RAW, {"ts_id": ts_id, "source_id": src_id})
            total_inserted += res.rowcount
    else:
        raise ValueError(f"Unsupported unit '{unit}' for series {src_id}")

debug(f"copied {total_inserted:,} facts into oil_network.timeseries_data")

n_data = pd.read_sql(text("SELECT COUNT(*) FROM oil_network.timeseries_data"), engine).iloc[0, 0]
n_series = pd.read_sql(text("""
    SELECT COUNT(DISTINCT timeseries_id) FROM oil_network.timeseries_data
"""), engine).iloc[0, 0]
debug(f"oil_network.timeseries_data: {n_data:,} rows across {n_series} series")

[17:20:26] copied 15,300 facts into oil_network.timeseries_data
[17:20:26] oil_network.timeseries_data: 68,793 rows across 137 series


## 3b. Combined inter-PADD movements
Build derived `combined_inter_padd_*_kbd` series summing pipeline + tanker/barge per direction, and bind them to the abstract inter-PADD flow variables.


## 3c. Derived foreign-supply inflows per region
Build `eia:foreign_supply_to_<region>_kbd` derived series, one per region (USA + 5 PADDs). For USA + PADDs 1-3, value = total imports - Canadian imports; for PADDs 4-5 with no Canadian-only observation, value = total imports directly. These series are the authoritative bindings for the `inflow__crude__<region>_view__foreign_supply` variables.


In [5]:
# FOREIGN_SUPPLY_DERIVED: build per-region non-Canadian foreign inflow series.
# Pattern parallels INTER_PADD_DIRECTIONS: catalogue row + SQL-built facts +
# variable_assignments bound to the derived series. The new EIA_SERIES tuples
# (FOREIGN_SUPPLY_USA / _P1 / ... / _P5) reference these series_ids; the actual
# data is built here.
FOREIGN_SUPPLY_REGIONS = [
    # (region_label, target_node, target_variable_id, total_imports_ts, canadian_inflow_ts_or_None)
    ("USA",   "usa_view",  "inflow__crude__usa_view__foreign_supply",   "eia:MCRIMUS2", "eia:MCRIMUSCA2"),
    ("P1",    "padd1_view", "inflow__crude__padd1_view__foreign_supply", "eia:MCRIPP12", "eia:MCRIPP1CA2"),
    ("P2",    "padd2_view", "inflow__crude__padd2_view__foreign_supply", "eia:MCRIPP22", "eia:MCRIPP2CA2"),
    ("P3",    "padd3_view", "inflow__crude__padd3_view__foreign_supply", "eia:MCRIPP32", "eia:MCRIPP3CA2"),
    ("P4",    "padd4_view", "inflow__crude__padd4_view__foreign_supply", "eia:MCRIPP42", "eia:MCRIPP4CA2"),
    ("P5",    "padd5_view", "inflow__crude__padd5_view__foreign_supply", "eia:MCRIPP52", "eia:MCRIPP5CA2"),
]

# Map region label -> derived ts_id (matches what the EIA_SERIES tuples expect).
DERIVED_TS_ID = {
    "USA": "eia:foreign_supply_to_usa_kbd",
    "P1":  "eia:foreign_supply_to_padd1_kbd",
    "P2":  "eia:foreign_supply_to_padd2_kbd",
    "P3":  "eia:foreign_supply_to_padd3_kbd",
    "P4":  "eia:foreign_supply_to_padd4_kbd",
    "P5":  "eia:foreign_supply_to_padd5_kbd",
}

# 1. Register catalogue rows for each derived series.
foreign_supply_cat = []
for region, node, _vid, total_ts, canadian_ts in FOREIGN_SUPPLY_REGIONS:
    derived_id = DERIVED_TS_ID[region]
    if canadian_ts is None:
        descr = f"{region} foreign-supply inflow = total imports {total_ts} (no Canadian-only series; equals total)"
    else:
        descr = f"{region} foreign-supply inflow = {total_ts} - {canadian_ts} (non-Canadian portion)"
    foreign_supply_cat.append({
        "timeseries_id":    derived_id,
        "name":             f"{region} non-Canadian foreign crude inflow (derived)",
        "description":      descr,
        "timeseries_type":  "inflow",
        "commodity":        "crude",
        "asset_id":         node,
        "related_asset_id": "foreign_supply",
        "source":           SOURCE,
        "unit":             "kbd",
        "attributes":       _json.dumps({
            "kind": "derived_foreign_supply",
            "total_imports_ts": total_ts,
            "canadian_inflow_ts": canadian_ts,
            "region": region,
        }),
    })

with engine.begin() as conn:
    conn.execute(UPSERT_TS, foreign_supply_cat)

# 2. Populate facts. For regions with Canadian breakdown: derived = total - Canadian.
#    For regions without: derived = total (just copy).
COPY_DIFF = text("""
    INSERT INTO oil_network.timeseries_data
        (timeseries_id, observation_date, saved_date, value)
    SELECT
        :derived_id,
        t.observation_date,
        GREATEST(t.saved_date, c.saved_date) AS saved_date,
        t.value - c.value AS value
    FROM (
        SELECT DISTINCT ON (observation_date) observation_date, saved_date, value
        FROM oil_network.timeseries_data
        WHERE timeseries_id = :total_ts
        ORDER BY observation_date, saved_date DESC
    ) t
    JOIN (
        SELECT DISTINCT ON (observation_date) observation_date, saved_date, value
        FROM oil_network.timeseries_data
        WHERE timeseries_id = :canadian_ts
        ORDER BY observation_date, saved_date DESC
    ) c USING (observation_date)
    ON CONFLICT (timeseries_id, observation_date, saved_date) DO UPDATE
        SET value = EXCLUDED.value
""")

COPY_PASSTHROUGH = text("""
    INSERT INTO oil_network.timeseries_data
        (timeseries_id, observation_date, saved_date, value)
    SELECT
        :derived_id, observation_date, saved_date, value
    FROM (
        SELECT DISTINCT ON (observation_date) observation_date, saved_date, value
        FROM oil_network.timeseries_data
        WHERE timeseries_id = :total_ts
        ORDER BY observation_date, saved_date DESC
    ) t
    ON CONFLICT (timeseries_id, observation_date, saved_date) DO UPDATE
        SET value = EXCLUDED.value
""")

n_fs_facts = 0
for region, _node, _vid, total_ts, canadian_ts in FOREIGN_SUPPLY_REGIONS:
    derived_id = DERIVED_TS_ID[region]
    with engine.begin() as conn:
        if canadian_ts is None:
            r = conn.execute(COPY_PASSTHROUGH, {"derived_id": derived_id, "total_ts": total_ts})
        else:
            r = conn.execute(COPY_DIFF, {"derived_id": derived_id, "total_ts": total_ts, "canadian_ts": canadian_ts})
        n_fs_facts += r.rowcount

debug(f"foreign_supply derived series: {len(FOREIGN_SUPPLY_REGIONS)} series, {n_fs_facts:,} facts written")


[17:20:26] foreign_supply derived series: 6 series, 804 facts written


In [6]:
# =========================================================================
# Build combined inter-PADD kbd series — pipeline + tanker/barge per direction
# Each direction's variable is bound to a derived `eia:combined_inter_padd_*_kbd`
# series whose values = sum(component_kbd) for the matching observation_date.
# Components are the auxiliary _kbd series from the dual-registration above.
# =========================================================================
INTER_PADD_DIRECTIONS = [
    # (from_padd, to_padd, [component_kbd_ts_ids], variable_id)
    ("P1", "P2", ["eia:MCRMPP2P11_kbd", "eia:MCRMTP2P11_kbd"],
        "outflow__crude__padd1_view__padd2_view"),
    ("P1", "P3", ["eia:MCRMPP3P11_kbd", "eia:MCRMTP3P11_kbd",
                  "eia:MCRMT1YP31_kbd", "eia:MCRMT1ZP31_kbd"],
        "outflow__crude__padd1_view__padd3_view"),
    ("P2", "P1", ["eia:MCRMPP1P21_kbd", "eia:MCRMTP1P21_kbd"],
        "outflow__crude__padd2_view__padd1_view"),
    ("P2", "P3", ["eia:MCRMPP3P21_kbd", "eia:MCRMTP3P21_kbd"],
        "outflow__crude__padd2_view__padd3_view"),
    ("P2", "P4", ["eia:MCRMPP4P21_kbd"],
        "outflow__crude__padd2_view__padd4_view"),
    ("P3", "P1", ["eia:MCRMP_R10-R30_1_kbd", "eia:MCRMTP1P31_kbd"],
        "outflow__crude__padd3_view__padd1_view"),
    ("P3", "P2", ["eia:MCRMPP2P31_kbd", "eia:MCRMTP2P31_kbd"],
        "outflow__crude__padd3_view__padd2_view"),
    ("P3", "P4", ["eia:MCRMP_R40-R30_1_kbd"],
        "outflow__crude__padd3_view__padd4_view"),
    ("P3", "P5", ["eia:MCRMTP5P31_kbd"],
        "outflow__crude__padd3_view__padd5_view"),
    ("P4", "P2", ["eia:MCRMPP2P41_kbd"],
        "outflow__crude__padd4_view__padd2_view"),
    ("P4", "P3", ["eia:MCRMPP3P41_kbd"],
        "outflow__crude__padd4_view__padd3_view"),
    ("P5", "P3", ["eia:MCRMTP3P51_kbd"],
        "outflow__crude__padd5_view__padd3_view"),
]

PADD_TO_VIEW_ASSET = {
    "P1": "padd1_view", "P2": "padd2_view",
    "P3": "padd3_view", "P4": "padd4_view",
    "P5": "padd5_view",
}
PADD_TO_REFINING_ASSET = {
    "P1": "padd1_view", "P2": "padd2_view",
    "P3": "padd3_view", "P4": "padd4_view",
    "P5": "padd5_view",
}

# 1. Register combined catalogue rows.
combined_rows = []
for src_padd, dst_padd, components, _var in INTER_PADD_DIRECTIONS:
    combined_id = f"eia:combined_inter_padd_{src_padd}_to_{dst_padd}_kbd"
    combined_rows.append({
        "timeseries_id":   combined_id,
        "name":            f"Inter-PADD {src_padd}->{dst_padd} total movement (pipeline + tanker/barge)",
        "description":     f"Sum of {len(components)} component _kbd series: {', '.join(components)}",
        "timeseries_type": "outflow",
        "commodity":       "crude",
        "asset_id":        PADD_TO_VIEW_ASSET[src_padd],
        "related_asset_id": PADD_TO_REFINING_ASSET[dst_padd],
        "source":          SOURCE,
        "unit":            "kbd",
        "attributes":      _json.dumps({
            "kind": "combined_inter_padd",
            "components": components,
            "from_padd": src_padd, "to_padd": dst_padd,
        }),
    })

with engine.begin() as conn:
    conn.execute(UPSERT_TS, combined_rows)

# 2. Populate facts: combined value = SUM of component _kbd values per (observation_date, saved_date).
#    Using the latest saved_date for each component, summed.
COMBINED_FACTS_SQL = text("""
    INSERT INTO oil_network.timeseries_data
        (timeseries_id, observation_date, saved_date, value)
    SELECT
        :combined_id, obs_date, MAX(saved_date) AS saved_date, SUM(val) AS value
    FROM (
        SELECT DISTINCT ON (timeseries_id, observation_date)
               timeseries_id,
               observation_date AS obs_date,
               saved_date,
               value AS val
        FROM oil_network.timeseries_data
        WHERE timeseries_id = ANY(:components)
        ORDER BY timeseries_id, observation_date, saved_date DESC
    ) latest
    GROUP BY obs_date
    ON CONFLICT (timeseries_id, observation_date, saved_date) DO UPDATE
        SET value = EXCLUDED.value
""")

total_combined_facts = 0
for src_padd, dst_padd, components, _var in INTER_PADD_DIRECTIONS:
    combined_id = f"eia:combined_inter_padd_{src_padd}_to_{dst_padd}_kbd"
    with engine.begin() as conn:
        res = conn.execute(COMBINED_FACTS_SQL, {"combined_id": combined_id, "components": components})
        total_combined_facts += res.rowcount

debug(f"Combined inter-PADD series: {len(INTER_PADD_DIRECTIONS)} catalogue rows, {total_combined_facts:,} facts written")

# 3. Bind the abstract flow variables to the combined series (overrides earlier pipeline-only bindings).
# Define UPSERT_ASSIGN inline so this cell is self-contained (the same SQL is used in cell "## 4. Bind authoritative variables").
UPSERT_ASSIGN = text("""
    INSERT INTO oil_network.variable_assignments
        (scenario_id, variable_id, effective_from, timeseries_id, formula, formula_inputs, notes)
    VALUES
        (:scenario_id, :variable_id, :effective_from, :timeseries_id, NULL, ARRAY[]::text[], :notes)
    ON CONFLICT (scenario_id, variable_id, effective_from) DO UPDATE SET
        timeseries_id  = EXCLUDED.timeseries_id,
        formula        = NULL,
        formula_inputs = ARRAY[]::text[],
        notes          = EXCLUDED.notes
""")
inter_padd_assignments = []
for src_padd, dst_padd, _comp, variable_id in INTER_PADD_DIRECTIONS:
    combined_id = f"eia:combined_inter_padd_{src_padd}_to_{dst_padd}_kbd"
    inter_padd_assignments.append({
        "scenario_id":    SCENARIO_ID,
        "variable_id":    variable_id,
        "effective_from": EFFECTIVE_FROM,
        "timeseries_id":  combined_id,
        "notes":          f"Total inter-PADD {src_padd}->{dst_padd}: sum of pipeline + tanker/barge components",
    })

with engine.begin() as conn:
    conn.execute(UPSERT_ASSIGN, inter_padd_assignments)

debug(f"Inter-PADD variable bindings: {len(inter_padd_assignments)} variables now bound to combined series")


[17:20:26] Combined inter-PADD series: 12 catalogue rows, 1,362 facts written
[17:20:26] Inter-PADD variable bindings: 12 variables now bound to combined series


## 4. Bind authoritative variables in `variable_assignments`

Only the **authoritative** rows from the mapping get a `variable_assignments` entry — auxiliary observations sit in the catalogue as observed data but do not participate in the mass balance (per principle 2.8). Each binding is `(scenario_id, variable_id, effective_from)` keyed; UPSERT keeps re-runs clean.

In [7]:
auth = df_map[df_map["role"] == "authoritative"]

UPSERT_ASSIGN = text("""
    INSERT INTO oil_network.variable_assignments
        (scenario_id, variable_id, effective_from, timeseries_id, formula, formula_inputs, notes)
    VALUES
        (:scenario_id, :variable_id, :effective_from, :timeseries_id, NULL, ARRAY[]::text[], :notes)
    ON CONFLICT (scenario_id, variable_id, effective_from) DO UPDATE SET
        timeseries_id  = EXCLUDED.timeseries_id,
        formula        = NULL,
        formula_inputs = ARRAY[]::text[],
        notes          = EXCLUDED.notes
""")

rows_assign = []
for row in auth.itertuples(index=False):
    # For mbbl sources, bind the variable to the derived kbd-rate version, not
    # the raw monthly volume. Variables in the framework are always rate (kbd)
    # for flow types. For mbbl_level (stock snapshot) and kbd sources, bind to
    # the raw ts_id; inventory variables are levels (MBBL), not rates.
    bound_ts_id = row.ts_id + "_kbd" if row.unit == "mbbl" else row.ts_id
    rows_assign.append({
        "scenario_id":    SCENARIO_ID,
        "variable_id":    row.variable_id,
        "effective_from": EFFECTIVE_FROM,
        "timeseries_id":  bound_ts_id,
        "notes":          (
            f"observed via EIA series {row.source_id}"
            + (" (kbd derived from monthly MBBL by per-row days conversion)" if row.unit == "mbbl" else "")
            + (" (stock level, MBBL snapshot)" if row.unit == "mbbl_level" else "")
        ),
    })

with engine.begin() as conn:
    conn.execute(UPSERT_ASSIGN, rows_assign)

n = pd.read_sql(text("""
    SELECT COUNT(*) FROM oil_network.variable_assignments
    WHERE scenario_id = :s AND timeseries_id IS NOT NULL
"""), engine, params={"s": SCENARIO_ID}).iloc[0, 0]
debug(f"variable_assignments (TS-bound, scenario={SCENARIO_ID}): {n} rows")

[17:20:26] variable_assignments (TS-bound, scenario=starter_us_crude_2015_2025): 115 rows


## 5. Verify

In [8]:
from IPython.display import display

print("-- Catalogue (oil_network.timeseries) --")
display(pd.read_sql(text("""
    SELECT timeseries_id, timeseries_type, asset_id, related_asset_id,
           attributes ->> 'role' AS role,
           attributes ->> 'scale_to_kbd' AS scale
    FROM oil_network.timeseries WHERE source = 'eia'
    ORDER BY role DESC, timeseries_id
"""), engine))

print("-- Sample fact: Permian-NM production for 2025 --")
display(pd.read_sql(text("""
    SELECT timeseries_id, observation_date, value
    FROM oil_network.timeseries_data
    WHERE timeseries_id = 'eia:MCRFPNM2'
      AND observation_date >= '2025-01-01' AND observation_date <= '2025-12-31'
    ORDER BY observation_date
"""), engine))

print("-- TS-bound variable_assignments under starter scope --")
display(pd.read_sql(text("""
    SELECT variable_id, timeseries_id, effective_from, notes
    FROM oil_network.variable_assignments
    WHERE scenario_id = :s AND timeseries_id IS NOT NULL
    ORDER BY variable_id
"""), engine, params={"s": SCENARIO_ID}))

print("-- Counts --")
display(pd.read_sql(text("""
    SELECT
        (SELECT COUNT(*) FROM oil_network.timeseries WHERE source='eia') AS catalogue_eia,
        (SELECT COUNT(*) FROM oil_network.timeseries_data) AS data_rows,
        (SELECT COUNT(DISTINCT timeseries_id) FROM oil_network.timeseries_data) AS data_series,
        (SELECT MIN(observation_date) FROM oil_network.timeseries_data) AS earliest,
        (SELECT MAX(observation_date) FROM oil_network.timeseries_data) AS latest,
        (SELECT COUNT(*) FROM oil_network.variable_assignments WHERE scenario_id=:s) AS assignments
"""), engine, params={"s": SCENARIO_ID}))

-- Catalogue (oil_network.timeseries) --


,timeseries_id,timeseries_type,asset_id,related_asset_id,role,scale
0,eia:combined_inter_padd_P1_to_P2_kbd,outflow,padd1_view,padd2_view,NaN,NaN
1,eia:combined_inter_padd_P1_to_P3_kbd,outflow,padd1_view,padd3_view,NaN,NaN
2,eia:combined_inter_padd_P2_to_P1_kbd,outflow,padd2_view,padd1_view,NaN,NaN
3,eia:combined_inter_padd_P2_to_P3_kbd,outflow,padd2_view,padd3_view,NaN,NaN
4,eia:combined_inter_padd_P2_to_P4_kbd,outflow,padd2_view,padd4_view,NaN,NaN
...,...,...,...,...,...,...
132,eia:MCRUA_R30_2,balancing_item,padd3_view,NaN,authoritative,1.0
133,eia:MCRUA_R40_2,balancing_item,padd4_view,NaN,authoritative,1.0
134,eia:MCRUA_R50_2,balancing_item,padd5_view,NaN,authoritative,1.0
135,eia:MCSSTUS1,inventory,spr_total,NaN,authoritative,1.0


-- Sample fact: Permian-NM production for 2025 --


,timeseries_id,observation_date,value
0,eia:MCRFPNM2,2025-01-01,2059.0
1,eia:MCRFPNM2,2025-01-01,2059.0
2,eia:MCRFPNM2,2025-01-01,2059.0
3,eia:MCRFPNM2,2025-01-01,2059.0
4,eia:MCRFPNM2,2025-02-01,2156.0
5,eia:MCRFPNM2,2025-02-01,2156.0
6,eia:MCRFPNM2,2025-02-01,2156.0
7,eia:MCRFPNM2,2025-02-01,2156.0
8,eia:MCRFPNM2,2025-03-01,2265.0
9,eia:MCRFPNM2,2025-03-01,2265.0


-- TS-bound variable_assignments under starter scope --


,variable_id,timeseries_id,effective_from,notes
0,balancing_item__crude__padd1_view,eia:MCRUA_R10_2,2015-01-01,observed via EIA series MCRUA_R10_2
1,balancing_item__crude__padd2_view,eia:MCRUA_R20_2,2015-01-01,observed via EIA series MCRUA_R20_2
2,balancing_item__crude__padd3_view,eia:MCRUA_R30_2,2015-01-01,observed via EIA series MCRUA_R30_2
3,balancing_item__crude__padd4_view,eia:MCRUA_R40_2,2015-01-01,observed via EIA series MCRUA_R40_2
4,balancing_item__crude__padd5_view,eia:MCRUA_R50_2,2015-01-01,observed via EIA series MCRUA_R50_2
...,...,...,...,...
110,production__crude__permian_nm,eia:MCRFPNM2,2015-01-01,observed via EIA series MCRFPNM2
111,production__crude__texas_state_view,eia:MCRFPTX2,2015-01-01,observed via EIA series MCRFPTX2
112,production__crude__usa_lower48_excl_gom_view,eia:PAPR48NGOM,2015-01-01,observed via EIA series PAPR48NGOM
113,production__crude__usa_view,eia:MCRFPUS2,2015-01-01,observed via EIA series MCRFPUS2


-- Counts --


,catalogue_eia,data_rows,data_series,earliest,latest,assignments
0,137,68793,137,2015-01-01,2027-12-01,975
